In [2]:
import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import xml.etree.ElementTree as ET
from tqdm.notebook import tqdm
import cv2
import torchvision.models as models

# --- CONFIGURATION ---
CONFIG = {
    # Updated paths for your new dataset
    'xml_root': r'E:\DATA\Annotations',
    'video_root': r'E:\DATA\Videos', 
    
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    
    # Dataset specific settings
    'target_label': 'bangla-tesla', # Updated based on your XML file
    'obs_len': 15,  
    'pred_len': 45, 
    
    # Hyperparameters
    'hidden_size': 256,
    'embed_size': 64,
    'batch_size': 32,
    'epochs': 15,
    'lr': 1e-3
}
print(f"✅ Config Loaded. Device: {CONFIG['device']}")
print(f"📂 XML Source: {CONFIG['xml_root']}")

✅ Config Loaded. Device: cpu
📂 XML Source: E:\DATA\Annotations


In [3]:
class IDDTrajectoryDataset(Dataset):
    def __init__(self, xml_root, obs_len=15, pred_len=45, target_label='bangla-tesla'):
        self.obs_len = obs_len
        self.pred_len = pred_len
        self.seq_len = obs_len + pred_len
        self.samples = []
        
        print(f"📂 Parsing XMLs for label: '{target_label}'...")
        xml_files = glob.glob(os.path.join(xml_root, '**', '*.xml'), recursive=True)
        
        if len(xml_files) == 0:
            print("⚠️ No XML files found! Check directory path.")
        
        for xml in tqdm(xml_files):
            try:
                tree = ET.parse(xml)
                root = tree.getroot()
                
                # Dynamic Resolution Parsing from XML Meta Data
                # This handles your 2592x1944 resolution automatically
                meta_size = root.find('meta').find('original_size')
                img_w = float(meta_size.find('width').text)
                img_h = float(meta_size.find('height').text)
                
                for track in root.findall('track'):
                    # Filter for specific label (bangla-tesla)
                    if track.attrib['label'] != target_label: continue
                    
                    track_data = [] # Will hold [cx, cy, w, h]
                    
                    # Sort boxes by frame attribute to ensure temporal order
                    boxes = sorted(track.findall('box'), key=lambda b: int(b.attrib['frame']))
                    
                    for box in boxes:
                        # Check if object is fully outside (optional, based on xml attribute)
                        if box.get('outside') == '1': continue

                        xtl, ytl = float(box.attrib['xtl']), float(box.attrib['ytl'])
                        xbr, ybr = float(box.attrib['xbr']), float(box.attrib['ybr'])
                        
                        # Dimensions
                        w = xbr - xtl
                        h = ybr - ytl
                        
                        # 1. Center Point
                        cx = (xtl + xbr) / 2
                        cy = (ytl + ybr) / 2
                        
                        # Normalization based on specific video resolution
                        track_data.append([cx/img_w, cy/img_h, w/img_w, h/img_h])
                    
                    track_data = np.array(track_data)
                    
                    # Skip if track is shorter than sequence length
                    if len(track_data) < self.seq_len: continue
                    
                    # Sliding window (stride 1 for max data, or 10 for less overlap)
                    stride = 10 
                    for i in range(0, len(track_data) - self.seq_len + 1, stride):
                        # Model Input: Only Center (cx, cy)
                        # We store W/H in output for reconstruction/metrics
                        obs = track_data[i : i+obs_len]
                        pred = track_data[i+obs_len : i+obs_len+pred_len]
                        
                        self.samples.append({
                            'obs_pos': obs[:, 0:2], # Center X, Center Y
                            'pred': pred            # cx, cy, w, h
                        })
            except Exception as e:
                # print(f"Error parsing {xml}: {e}") # Uncomment to debug bad files
                pass
                
    def __len__(self): return len(self.samples)
    
    def __getitem__(self, idx):
        item = self.samples[idx]
        return (
            torch.tensor(item['obs_pos'], dtype=torch.float32), # Input (15, 2)
            torch.tensor(item['pred'], dtype=torch.float32)     # Target (45, 4)
        )

# Re-create Dataset
dataset = IDDTrajectoryDataset(
    CONFIG['xml_root'], 
    obs_len=CONFIG['obs_len'], 
    pred_len=CONFIG['pred_len'],
    target_label=CONFIG['target_label']
)
print(f"✅ Dataset Created: {len(dataset)} sequences.")

📂 Parsing XMLs for label: 'bangla-tesla'...


  0%|          | 0/6 [00:00<?, ?it/s]

✅ Dataset Created: 2156 sequences.


In [4]:
class PIETrajNet(nn.Module):
    def __init__(self, input_size=2, output_size=4, hidden_size=256, embed_size=64):
        super(PIETrajNet, self).__init__()
        
        # Embedding for coordinates (Input is [cx, cy])
        self.embed = nn.Linear(input_size, embed_size)
        self.relu = nn.ReLU()
        
        # Encoder LSTM
        self.encoder = nn.LSTM(embed_size, hidden_size, batch_first=True)
        
        # Decoder LSTM
        self.decoder = nn.LSTM(embed_size, hidden_size, batch_first=True)
        
        # Output Layer (Output is [cx, cy, w, h])
        self.fc = nn.Linear(hidden_size, output_size)
        
    def forward(self, obs, pred_len):
        # obs: [Batch, Obs_Len, 2]
        
        # 1. Encode
        # -----------------------------
        emb = self.relu(self.embed(obs)) # Embed spatial coordinates
        _, (h, c) = self.encoder(emb)    # Get hidden state from sequence
        
        # 2. Decode (Autoregressive)
        # -----------------------------
        outputs = []
        
        # Start decoding with the last observed position [cx, cy]
        curr_input = obs[:, -1, :].unsqueeze(1) # Shape: [Batch, 1, 2]
        
        for _ in range(pred_len):
            # Embed current input
            curr_emb = self.relu(self.embed(curr_input))
            
            # Run Decoder LSTM step
            out, (h, c) = self.decoder(curr_emb, (h, c))
            
            # Predict [cx, cy, w, h]
            pred_step = self.fc(out) # Shape: [Batch, 1, 4]
            
            outputs.append(pred_step)
            
            # Prepare input for next step:
            # We take the predicted [cx, cy] part (index :2) to feed back in
            curr_input = pred_step[:, :, :2] 
            
        return torch.cat(outputs, dim=1) # Returns [Batch, Pred_Len, 4]

# Initialize Model
model = PIETrajNet(
    input_size=2, # x, y
    output_size=4, # x, y, w, h
    hidden_size=CONFIG['hidden_size'],
    embed_size=CONFIG['embed_size']
).to(CONFIG['device'])

print(f"✅ Model Initialized on {CONFIG['device']}")
# Quick Test with dummy data to ensure shapes match
dummy_input = torch.rand(CONFIG['batch_size'], CONFIG['obs_len'], 2).to(CONFIG['device'])
dummy_output = model(dummy_input, CONFIG['pred_len'])
print(f"🔬 Shape Check - Input: {dummy_input.shape} -> Output: {dummy_output.shape}")

✅ Model Initialized on cpu
🔬 Shape Check - Input: torch.Size([32, 15, 2]) -> Output: torch.Size([32, 45, 4])


In [5]:
# 1. Split Data into Train and Validation
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_set, val_set = torch.utils.data.random_split(dataset, [train_size, val_size])

# 2. Create DataLoaders
train_loader = DataLoader(train_set, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(val_set, batch_size=CONFIG['batch_size'], shuffle=False)

# 3. Setup Optimizer and Loss
optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'])
criterion = nn.MSELoss() # Mean Squared Error Loss

print(f"🚀 Starting Training on {len(train_set)} samples, Validating on {len(val_set)}...")

# 4. Training Loop
for epoch in range(CONFIG['epochs']):
    model.train()
    train_loss = 0
    
    for obs, target in tqdm(train_loader, leave=False, desc=f"Epoch {epoch+1}"):
        # Move data to GPU/CPU
        obs = obs.to(CONFIG['device'])       # Input: [Batch, 15, 2] (x, y)
        target = target.to(CONFIG['device']) # Target: [Batch, 45, 4] (x, y, w, h)
        
        optimizer.zero_grad()
        
        # Forward Pass
        preds = model(obs, CONFIG['pred_len']) # Output: [Batch, 45, 4]
        
        # Calculate Loss
        # We compare the predicted (x,y,w,h) directly against the ground truth
        loss = criterion(preds, target)
        
        # Backward Pass
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
    # 5. Validation Loop
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for obs, target in val_loader:
            obs = obs.to(CONFIG['device'])
            target = target.to(CONFIG['device'])
            
            preds = model(obs, CONFIG['pred_len'])
            val_loss += criterion(preds, target).item()
            
    # Log metrics
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    print(f"Epoch {epoch+1}/{CONFIG['epochs']} | Train Loss: {avg_train_loss:.6f} | Val Loss: {avg_val_loss:.6f}")

# 6. Save the trained model
torch.save(model.state_dict(), "bangla_tesla_traj_model.pth")
print("💾 Model Saved as 'bangla_tesla_traj_model.pth'")

🚀 Starting Training on 1724 samples, Validating on 432...


Epoch 1:   0%|          | 0/54 [00:00<?, ?it/s]

Epoch 1/15 | Train Loss: 0.020974 | Val Loss: 0.010016


Epoch 2:   0%|          | 0/54 [00:00<?, ?it/s]

Epoch 2/15 | Train Loss: 0.010084 | Val Loss: 0.009533


...

Epoch 13/15 | Train Loss: 0.003074 | Val Loss: 0.003237


Epoch 14:   0%|          | 0/54 [00:00<?, ?it/s]

Epoch 14/15 | Train Loss: 0.002611 | Val Loss: 0.002783


Epoch 15:   0%|          | 0/54 [00:00<?, ?it/s]

Epoch 15/15 | Train Loss: 0.002874 | Val Loss: 0.002966
💾 Model Saved as 'bangla_tesla_traj_model.pth'


In [6]:
import pickle
import numpy as np
import torch

def calculate_metrics_1500ms(model, loader):
    model.eval()
    
    mse_traj_list = []    # Average Squared Error over the full path
    cmse_final_list = []  # Center Squared Error at final frame (1.5s)
    cfmse_final_list = [] # Center + Foot Squared Error at final frame (1.5s)
    
    # ---------------------------------------------------------
    # RESOLUTION CONFIG (Based on your XML: 2592 x 1944)
    # ---------------------------------------------------------
    W_px, H_px = 2592, 1944
    
    with torch.no_grad():
        for obs, target in loader:
            obs = obs.to(CONFIG['device'])
            target = target.cpu().numpy() # Shape: [Batch, 45, 4] (cx, cy, w, h)
            
            # Predict full trajectory
            # Model output is now [Batch, 45, 4]
            preds = model(obs, CONFIG['pred_len']).cpu().numpy() 
            
            # --- Un-normalize Coordinates to Pixels ---
            # Create copies to avoid modifying original tensors
            pred_seq = np.zeros_like(preds)
            gt_seq = np.zeros_like(target)
            
            # Scale X and Width by Width
            pred_seq[:, :, 0] = preds[:, :, 0] * W_px # cx
            pred_seq[:, :, 2] = preds[:, :, 2] * W_px # w
            gt_seq[:, :, 0] = target[:, :, 0] * W_px
            gt_seq[:, :, 2] = target[:, :, 2] * W_px
            
            # Scale Y and Height by Height
            pred_seq[:, :, 1] = preds[:, :, 1] * H_px # cy
            pred_seq[:, :, 3] = preds[:, :, 3] * H_px # h
            gt_seq[:, :, 1] = target[:, :, 1] * H_px
            gt_seq[:, :, 3] = target[:, :, 3] * H_px
            
            # --- 1. MSE (Trajectory Average) ---
            # We compare only centers (index 0 and 1) for standard trajectory MSE
            pred_pos = pred_seq[:, :, 0:2]
            gt_pos = gt_seq[:, :, 0:2]
            
            # Squared Euclidean Distance: (x-x)^2 + (y-y)^2
            # Mean over time (axis 1), Sum over coords (axis 2)
            traj_mse = np.mean(np.sum((pred_pos - gt_pos)**2, axis=2), axis=1)
            mse_traj_list.extend(traj_mse)

            # --- 2. C-MSE (Final Frame Center) ---
            # Extract position at the last predicted time step (t=45)
            pred_c_end = pred_pos[:, -1, :]
            gt_c_end = gt_pos[:, -1, :]
            
            c_mse = np.sum((pred_c_end - gt_c_end)**2, axis=1)
            cmse_final_list.extend(c_mse)
            
            # --- 3. CF-MSE (Final Frame Center + Foot) ---
            # Since model predicts Height, we calculate Foot location based on Prediction
            
            # Ground Truth Foot: cy + (h/2)
            gt_h_end = gt_seq[:, -1, 3]
            gt_foot_y = gt_c_end[:, 1] + (gt_h_end / 2.0)
            gt_foot = np.stack([gt_c_end[:, 0], gt_foot_y], axis=1)
            
            # Predicted Foot: cy + (predicted_h / 2)
            pred_h_end = pred_seq[:, -1, 3]
            pred_foot_y = pred_c_end[:, 1] + (pred_h_end / 2.0)
            pred_foot = np.stack([pred_c_end[:, 0], pred_foot_y], axis=1)
            
            # Foot MSE
            foot_mse = np.sum((pred_foot - gt_foot)**2, axis=1)
            
            # CF-MSE = Center MSE + Foot MSE
            cfmse_final_list.extend(c_mse + foot_mse)

    final_mse = np.mean(mse_traj_list)
    final_cmse = np.mean(cmse_final_list)
    final_cfmse = np.mean(cfmse_final_list)
    
    return final_mse, final_cmse, final_cfmse

# Run Metrics on Validation Set
print("📊 Calculating Metrics...")
mse, c_mse, cf_mse = calculate_metrics_1500ms(model, val_loader)

print(f"\n✅ Bangla-Tesla Dataset Results (Resolution 2592x1944):")
print(f"   MSE (Avg Trajectory): {mse:.2f}")
print(f"   C-MSE (Center @ 1.5s):  {c_mse:.2f}")
print(f"   CF-MSE (Center+Foot @ 1.5s): {cf_mse:.2f}")

# Save Results
results = {'MSE': mse, 'C-MSE': c_mse, 'CF-MSE': cf_mse}
with open('result_bangla_tesla.pkl', 'wb') as f:
    pickle.dump(results, f)
print("\n📝 Results saved to 'result_bangla_tesla.pkl'")

📊 Calculating Metrics...

✅ Bangla-Tesla Dataset Results (Resolution 2592x1944):
   MSE (Avg Trajectory): 19385.38
   C-MSE (Center @ 1.5s):  51930.15
   CF-MSE (Center+Foot @ 1.5s): 113628.50

📝 Results saved to 'result_bangla_tesla.pkl'
